# Binary classification with random forest

This notebook walks through the full loop for one classifier: prepare the data, build a `TabularDataset` from a YAML config, train with `SimpleTrainer`, evaluate, and cross-validate.

The same steps are then repeated further down with a torch MLP instead of a random forest. The only thing that differs between the two runs is the config file — the notebook code is essentially identical, which is the point of the protocol-based design.

**Before you run this notebook, delete the results from previous runs of any example notebooks to avoid getting 'file exists' errors.**

## Build the dataset and split it

The raw Cloudy `.dat` grids carry no label: which class a row belongs to is only encoded in the file name (`AGN` vs `POPSTAR`). `to_xy` reads its labels from a column, so the cell below turns that filename information into an explicit `source` column (0 = AGN, 1 = star-forming) and writes each grid out as a plain comma-separated `.csv`.

That is also why the dataset config uses `data_format: csv`: Hugging Face Datasets loads the generated comma-separated files instead of the whitespace-separated originals.

We use a 'medal' scheme for data quality assignment: `bronze` is raw data, `silver` is generally preprocessed data, and `gold` is data preprocessed for particular machine learning training runs, ready and waiting. 
Here, our preprocessing moves data from `bronze` to `silver`. 

For this training run, we reuse the bronze-to-silver pre-transform directly instead of reading pre-existing silver CSV files. `TabularDataset` parses the two tracked bronze grids, maps their source strings to `source_label`, and caches the prepared Arrow dataset in `data/silver/default`. The resulting dataset is immediately split and passed to `SimpleTrainer`; no intermediate CSV-loading step is required.

`random_split` returns `torch.utils.data.Subset` objects — no data is copied, each one just holds the dataset plus a list of row positions. `to_xy` understands those, so a split can be materialised on its own.


In [ ]:
import torch
import yaml
from torch.utils.data import random_split
from pprint import pprint

from GalaxySpectrumClassifier import TabularDataset, to_xy

with open("../configs/binary_classsifier_simple_example.yaml", "r") as f:
    config = yaml.safe_load(f)


def encode_source(row):
    """Map raw Cloudy source strings to contiguous classifier labels."""
    labels = {"AGN": 0, "HII": 1}
    try:
        return {"source_label": labels[row["source"]]}
    except KeyError as error:
        raise ValueError(f"Unknown source label: {row['source']!r}") from error


dataset = TabularDataset(
    data_format="csv",
    label_columns="source_label",
    pre_transform="__main__.encode_source",
    # Replace the raw string label so Arrow records an integer target schema.
    pre_transform_kwargs={"remove_columns": ["source"]},
    hf_dataset_kwargs={
        "data_files": [
            "../data/bronze/default/C17_AGN_alpha08_efrac02_CNfix.dat",
            "../data/bronze/default/C17_POPSTAR_1myr.dat",
        ],
        "comment": "#",
        "sep": r"\s+",
        "cache_dir": "../data/silver/default",
    },
)
train_dataset, test_dataset = random_split(
    dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)

# Train model 

The `trainer` block of the same YAML builds the estimator: a `RandomForestClassifier` wrapped in `CalibratedClassifierCV`, so the calibrator is what actually gets fitted and evaluated.

`fit` calls `to_xy(train_dataset)` internally. `label_columns: source` in the dataset config makes each retrieved sample split the `source` column into `y` and everything else into `X` before the estimator's `.fit()` receives the materialized arrays.

`save_snapshot` writes two files under the trainer's `output_path`: `config.yaml` (everything needed to rebuild the trainer) and `model.skops` (the fitted estimator).

In [ ]:
import yaml

from GalaxySpectrumClassifier import SimpleTrainer

with open("../configs/binary_classsifier_simple_example.yaml", "r") as f:
    config = yaml.safe_load(f)
trainer = SimpleTrainer.from_config(config["trainer"])

trainer.fit(train_dataset)
trainer.save_snapshot("trained_random_forest")

# Test model

`load_snapshot` rebuilds the trainer from the saved config and loads the fitted model back in, so this cell could be run in a fresh session without retraining. By default it reuses the saved output directory; pass `save_to=...` to give subsequent outputs a new base directory.

`evaluate` scores the model with every metric listed in the config and returns a plain `{name: score}` dict. Metrics marked `needs_proba: true` are scored against `predict_proba()[:, 1]` instead of the hard predictions; that is what `task: binary-classification` controls.

In [ ]:
from GalaxySpectrumClassifier import SimpleTrainer

trainer = SimpleTrainer.load_snapshot(
    "../training/binaryclassifier_simple_example/trained_random_forest"
)

In [ ]:
import pandas as pd

from GalaxySpectrumClassifier import SimpleTrainer

test_results = trainer.evaluate(test_dataset)
test_results = pd.DataFrame.from_dict(
    data=[
        test_results,
    ]
)
test_results.to_csv(
    trainer.output_path / "trained_random_forest/test_results.csv", index=False
)
test_results

that these metrics are so pathologically high is an artifact of the data selection, not really of the quality of the classifier as such

## Random permutations cross-validation
this is used to get a handle on the influence of sample choice on the model. Here, because of the aforementioned reason, the results are always the same and only the losses are a little bit different. 

The loop below does what `cross_val_score` would do, but keeps the trainer in the picture so the configured metrics and the calibrator are used rather than a single default score.

`StratifiedKFold` only supplies row positions; the actual splits are built as `Subset(dataset, idx)`, so no data is copied per fold. Each `trainer.fit` call refits the estimator from scratch — sklearn resets its learned state on every `.fit()` — so the folds are independent, and the trainer holds the last fold's model once the loop finishes.

`to_xy` is called here only to obtain `y` for the stratification; the fitting itself goes through the subsets.

In [ ]:
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)

X, y = to_xy(dataset)

test_results = []
for i, (train_idx, test_idx) in enumerate(
    kfold.split(
        X,
        y,
    )
):
    trainer = SimpleTrainer.load_snapshot(
        "../training/binaryclassifier_simple_example/trained_random_forest",
        save_to=f"../training/binaryclassifier_simple_example/trained_random_forest/run_{i}",
    )
    train_dataset = torch.utils.data.Subset(dataset, train_idx)
    test_dataset = torch.utils.data.Subset(dataset, test_idx)

    t = trainer.evaluate(test_dataset)
    tdf = pd.DataFrame.from_dict(
        data=[
            t,
        ]
    )

    test_results.append(tdf)

print(pd.concat(test_results))

# Binary classification with torch-based 2-layer multilayer perceptron

Everything below repeats the same sequence with a neural network. The notebook code is unchanged apart from the config path — `SimpleTrainer` has no special handling for torch models.

What makes that work is skorch: `skorch.NeuralNetClassifier` gives a torch module the same `fit`/`predict`/`predict_proba` interface an sklearn estimator has, including the epoch loop, batching and optimizer. The config points it at `torchvision.ops.MLP` with `hidden_channels: [64, 2]`, which is `Linear(18, 64) -> ReLU -> Linear(64, 2)`, and sets `CrossEntropyLoss` because the module emits raw logits.

The calibrator wraps the network exactly as it wrapped the forest.

In [ ]:
import torch
import yaml
from torch.utils.data import random_split

from GalaxySpectrumClassifier import SimpleTrainer, TabularDataset


def convert_types(batch):
    batch = dict(batch)
    for name, values in batch.items():
        cast = int if name == "source" else float
        batch[name] = [cast(value) for value in values]
    return batch


with open("../configs/binary_classsifier_simple_example_torch.yaml", "r") as f:
    config = yaml.safe_load(f)

dataset = TabularDataset.from_config(config["dataset"])
train_dataset, test_dataset = random_split(
    dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)

trainer = SimpleTrainer.from_config(config["trainer"])

trainer.fit(train_dataset)
trainer.save_snapshot("trained_mlp")

Note that `SimpleTrainer` does one non-resumable `.fit()` call, so the epochs configured for the MLP run entirely inside skorch and are invisible from here — there is no per-epoch loss to inspect, no checkpointing and no early stopping through the trainer. That is the limit of this class; an epoch-based trainer is the planned counterpart for torch models.

In [ ]:
import pandas as pd

from GalaxySpectrumClassifier import SimpleTrainer

trainer = SimpleTrainer.load_snapshot(
    "../training/binaryclassifier_simple_example_torch/trained_mlp"
)
test_results = trainer.evaluate(test_dataset)
test_results = pd.DataFrame.from_dict(
    data=[
        test_results,
    ]
)
test_results.to_csv(trainer.output_path / "trained_mlp/test_results.csv", index=False)
test_results

## apply StratifiedKFold with the torch model 

Identical to the cross-validation loop above, only with the MLP trainer. Each fold retrains the network from scratch, so this cell is considerably slower than the random forest version.

In [ ]:
import torch
import yaml
from sklearn.model_selection import StratifiedKFold

from GalaxySpectrumClassifier import SimpleTrainer, TabularDataset, to_xy

with open("../configs/binary_classsifier_simple_example_torch.yaml", "r") as f:
    config = yaml.safe_load(f)

dataset = TabularDataset.from_config(config["dataset"])
kfold = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)

X, y = to_xy(dataset)
res = []
for i, (train_idx, test_idx) in enumerate(
    kfold.split(
        X,
        y,
    )
):
    trainer = SimpleTrainer.load_snapshot(
        "../training/binaryclassifier_simple_example_torch/trained_mlp",
        save_to=f"../training/binaryclassifier_simple_example_torch/trained_mlp/run_{i}",
    )
    train_dataset = torch.utils.data.Subset(dataset, train_idx)
    test_dataset = torch.utils.data.Subset(dataset, test_idx)

    test_results = trainer.evaluate(test_dataset)
    test_results = pd.DataFrame.from_dict(
        data=[
            test_results,
        ]
    )

    res.append(test_results)

print(pd.concat(res))

# `sklearn` Regression models

Let's try a regression model, too, for good measure. Here, we build a support-vector maching, using scikit-learn's `SVR` variant. We try and make the class column we built above called 'source' the regression target here for demonstration purposes. 

In [ ]:
import pandas as pd
import torch
import yaml
from torch.utils.data import random_split

from GalaxySpectrumClassifier import SimpleTrainer, TabularDataset, to_xy

with open("../configs/regression_svm_simple_example.yaml", "r") as f:
    config = yaml.safe_load(f)

pprint("config file: ")
pprint(config)

dataset = TabularDataset.from_config(config["dataset"])
train_dataset, test_dataset = random_split(
    dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)

trainer = SimpleTrainer.from_config(config["trainer"])

trainer.fit(train_dataset)

test_results = trainer.evaluate(test_dataset)
test_results = pd.DataFrame.from_dict(
    data=[
        test_results,
    ]
)

trainer.save_snapshot("svr_model")
test_results.to_csv(trainer.output_path / "test_results.csv", index=False)
test_results